In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pycontrails import Fleet

In [2]:
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [3]:
# TODO fuel burn? nox? depends on if I want to have cruise only
cols = ["fuel_burn", "nox", "NOx", "O3", "CH4", "H2O"]
cols = ["NOx", "O3", "CH4", "H2O"]

from cane.utils import mask_by_marker

mask_by_marker(fleetf, cols)
mask_by_marker(fleeto, cols)

dff = fleetf.dataframe
dfo = fleeto.dataframe

from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, cols, bounds)
mask_by_validity_range(fleeto, cols, bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [4]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

dff["NOx_H2O"] = dff["NOx"] + dff["H2O"]
dfo["NOx_H2O"] = dfo["NOx"] + dfo["H2O"]

In [5]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [6]:
cols = ["fuel_burn_diff", "nox_diff", "co2_diff", "CO2_diff", "NOx_diff", "CoCiP_diff", "H2O_diff", "CO2_CoCiP_diff",
        "NOx_H2O_diff",
        "Total_diff"]

tmp = my_diff[cols].sum().rename("diff_per_flight") / my_diff.shape[0]  # changes in [kg] per rerouted flight
tmp = tmp.reset_index()
# tmp["diff_per_flight"] = tmp["diff_per_flight"]  #.round(2)

In [7]:
cols = ["fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "H2O", "CO2_CoCiP", "NOx_H2O", "Total"]

rels = my_diff[[f"{col}_diff" for col in cols]].sum().values / my_diff[
    [f"{col}_filed" for col in cols]].sum().values  # relative change

tmp["reldiff_per_flight"] = rels
print(my_diff.shape)
# tmp
for i, r in tmp.iterrows():
    print(r["index"], round(r.diff_per_flight, 0), f"{round(r.reldiff_per_flight * 100, 1)}%", sep="\t")

(4112, 42)
fuel_burn_diff	357.0	1.2%
nox_diff	13.0	2.5%
co2_diff	1129.0	1.2%
CO2_diff	1129.0	1.2%
NOx_diff	380.0	1.1%
CoCiP_diff	-26932.0	-43.5%
H2O_diff	-33.0	-1.2%
CO2_CoCiP_diff	-25830.0	-16.5%
NOx_H2O_diff	346.0	1.0%
Total_diff	-25483.0	-13.3%
